In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

file_name = dbutils.widgets.get("file_name")
table = dbutils.widgets.get("table") 

pandas_path = f"/Volumes/{catalog}/{schema}/{volume}/{file_name}"
print(f"Ruta volumen: {pandas_path}")

In [0]:
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
column_mapping = {
    'RESPUESTA': 'respuesta', 
    'CONSUMO_CRITICADO': 'consumo_criticado', 
    'SERVICIO': 'servicio',
    'CATEGORIA': 'categoria', 
    'NIVEL_TENSION': 'nivel_tension', 
    'ESTRATO': 'estrato', 
    'LOCALIDAD': 'localidad', 
    'FUNCION_ANALISIS': 'funcion_analisis',
    'CALIFICACION': 'calificacion',
    'OBS_LECTURA': 'obs_lectura',
    'PERIODICIDAD': 'periodicidad'
}

In [0]:
import pandas as pd

df_pandas = pd.read_excel(pandas_path)

print(df_pandas.columns)

In [0]:
mapeo_tipo_servicio = {
    '701-ENERGÍA MDO REGULADO': 'Energia',
    '101-AGUA POTABLE': 'Agua',
    '103-ALCANTARILLADO': 'Alcantarillado',
    '501-GAS NATURAL REGULADO': 'Gas',
    '1007-ALUMBRADO PÚBLICO MEDIDO': 'Energia',
    '8000-AGUA POTABLE OCC.': 'Agua',
    '8003-AGUA POTABLE URABA': 'Agua',
    '7505-GAS NATURAL COMPRIMIDO (GNC)': 'Gas',
    '240-AGUA POTABLE MALAMBO': 'Agua',
    '249-AGUA POTABLE DE RIONEGRO ANT': 'Agua',
    '1702-MOVILIDAD ELÉCTRICA CARGA INTE': 'Energia'
}


df_pandas['tipo_servicio'] = df_pandas['SERVICIO'].map(mapeo_tipo_servicio)
df_pandas.drop(columns=['SERVICIO'], inplace=True)

# Intercambiar el valor de la variable respuesta
df_pandas['RESPUESTA'] = 1 - df_pandas['RESPUESTA']

#df_pandas['id'] = df_pandas.index + 1
#cols = df_pandas.columns.tolist()
#cols.remove('id')
#new_cols_order = ['id'] + cols
#df_pandas = df_pandas[new_cols_order]

display(df_pandas.head())

In [0]:
df_spark = spark.createDataFrame(df_pandas)

for old_name, new_name in column_mapping.items():
    df_spark = df_spark.withColumnRenamed(old_name, new_name)

In [0]:
(df_spark.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(table)
)

In [0]:
%sql
select *
from ordenes